# Lab 2 — Async/Await and File Upload for a Document Q&A Pipeline

Difficulty: Intermediate | ~35-40 min 

### Step 1: Install Dependencies

We install the exact pinned versions of every library this lab needs. Run this cell first so everything is available for the rest of the notebook.

In [ ]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 google-genai==1.29.0 python-dotenv==1.2.3 python-multipart numpy

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   --------------------------- ------------ 2.6/3.8 MB 15.1 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 7.4 MB/s  0:00:00

   ---------------------------------------- 0/9 [websockets]
   ---- ----------------------------------- 1/9 [tenacity]
   ------------- -------------------------- 3/9 [pyasn1]
   ----------------- ---------------------- 4/9 [pyasn1-modules]
   ----------------- ---------------------- 4/9 [pyasn1-modules]
   ----------------- ---------------------- 4/9 [pyasn1-modules]
  Attempting uninstall: httpx
   ----------------- ---------------------- 4/9 [pyasn1-modules]
    Found existing installation: httpx 0.27.2
   ----------------- ---------------------- 4/9 [pyasn1-modules]
    Uninstalling httpx-0.27.2:
   ----------------- ---------------------- 4/9 [pyasn1-modules]
      Successfully uninstalle


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Imports and Client Setup

We import the standard library modules we need, load the Google API key from a `.env` file using `python-dotenv`, and initialize the Google GenAI client. The client auto-detects the `GOOGLE_API_KEY` environment variable when no explicit key is passed.

In [40]:
import time
import os
import asyncio
from dotenv import load_dotenv
from fastapi import FastAPI, UploadFile
from fastapi.testclient import TestClient
from google import genai
import numpy as np

# Load GOOGLE_API_KEY from .env file in the current directory
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    api_key = input("Enter your Google API key: ")

# The client picks up GOOGLE_API_KEY from the environment automatically
client = genai.Client(api_key=api_key)

### Step 4: Text Chunking

Before embedding, we split the document into fixed-size character chunks. This is a simple but effective approach — each chunk becomes one embedding vector. We use a `chunk_size` of 200 characters with no overlap, keeping it dependency-free.

In [18]:
def chunk_text(text, chunk_size = 200):
    """Split text into fixed-size chunks with no overlap."""
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks

### Step 5: Embed a Single Chunk

This is the core API call. We use the **async** client (`client.aio`) to embed a single text chunk. The hint tells the model this text will be used for retrieval (as opposed to a search query), which can improve embedding quality.

In [19]:
async def embed(text):
    """Embed text using the async Google GenAI client."""
    # Use the async client — this is what makes real concurrency possible
    response = await client.aio.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
    )
    # response.embeddings is a list; for one input, take the first (and only) item
    return response.embeddings[0].values

### Step 6: Sequential Embedding

This function embeds every chunk **one at a time**. We wrap the loop with `time.time()` to measure the total time. Even though the function is `async def`, using `await` inside a `for` loop still runs calls sequentially.

In [ ]:
async def embed_all_sequential(chunks):
    """Embed chunks one at a time. Returns (embeddings, elapsed_seconds)."""
    embeddings = []
    start = time.time()
    for chunk in chunks:
        embedding = await embed(chunk)
        embeddings.append(embedding)
    elapsed = time.time() - start
    return embeddings, elapsed

### Step 7: Concurrent Embedding

This is the key difference. `asyncio.gather()` **starts all the embed calls at once** before waiting on any of them. All 8 network requests are in flight simultaneously, so the total time is close to the duration of the slowest single call, not the sum of all 8.

In [ ]:
async def embed_all_concurrent(chunks):
    """Embed all chunks concurrently. Returns (embeddings, elapsed_seconds)."""
    start = time.time()
    # Launch all embed calls at once — they all run in parallel
    embeddings = await asyncio.gather(*[embed(c) for c in chunks])
    elapsed = time.time() - start
    return list(embeddings), elapsed

### Step 8: Retrieve Top-K Chunks

Given a query embedding and a list of stored chunks (each with their precomputed embeddings), we score every chunk against the query using cosine similarity and return the top `k` matches. This is the retrieval step of a RAG pipeline.

In [28]:
def cosine_similarity(a, b):
    similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    return similarity

In [23]:
def retrieve_top_k(query_embedding, stored_chunks, k=3):
    """Return the k chunks most similar to the query embedding."""
    scored = []
    for chunk in stored_chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        scored.append((score, chunk))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [chunk for score, chunk in scored[:k]]

### Step 9: Generate an Answer

We build a prompt that combines the retrieved context chunks with the user's question, then send it to the LLM. The model reads the context and answers the question based on it — this is the generation step of a RAG pipeline.

In [24]:
async def generate_answer(question, context_chunks):
    """Generate an answer using retrieved context chunks."""

    context = "\n\n".join([c["text"] for c in context_chunks])
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer based on the context:"

    response = await client.aio.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

### Step 10: Sequential Upload Endpoint

This endpoint accepts a file upload, chunks the text, and embeds each chunk.

In [33]:
# Simple in-memory store — each entry holds a text chunk and its embedding vector
document_store = []

app = FastAPI()

@app.post("/upload/")
async def upload(file: UploadFile, sequential: bool = False):
    """Upload a file and embed its chunks one at a time."""

    text = (await file.read()).decode("utf-8")
    chunks = chunk_text(text)
    
    if sequential:
        embeddings, elapsed = await embed_all_sequential(chunks)
    else:
        embeddings, elapsed = await embed_all_concurrent(chunks)

    # Store each chunk paired with its embedding
    for chunk_text_val, emb in zip(chunks, embeddings):
        document_store.append({"text": chunk_text_val, "embedding": emb})
        
    return {"chunk_count": len(chunks), "elapsed_seconds": round(elapsed, 2)}

### Step 11: Query Endpoint

The query endpoint ties everything together: it embeds the user's question, retrieves the most relevant stored chunks, passes them as context to the LLM, and returns the answer along with the source texts.

In [36]:
@app.post("/query")
async def query(question: str):
    """Answer a question using stored document chunks."""
    q_emb = await embed(question)
    top_chunks = retrieve_top_k(q_emb, document_store, k=5)

    answer = await generate_answer(question, top_chunks)
    
    return {"answer": answer, "sources": [c["text"] for c in top_chunks]}

### Step 12: Demo — Sequential vs Concurrent Upload + Query

We use `TestClient` as a context manager (`with` statement) to test all endpoints. This ensures the event loop is managed correctly across multiple async requests. We upload to the endpoint sequentially and concurrently with the same text — the timing difference is the core lesson. Finally, we test the full RAG pipeline with a question.

In [38]:
# Read the sample document from disk
with open("sample_text_file.txt", "rb") as f:
    file_bytes = f.read()

with TestClient(app) as test_client:
    # Sequential: each await blocks the next call
    files = {"file": ("sample_text_file.txt", file_bytes, "text/plain")}
    seq_data = test_client.post("/upload/?sequential=true", files=files).json()
    print(f"Sequential: {seq_data['chunk_count']} chunks in {seq_data['elapsed_seconds']}s")

    # Concurrent: all calls launched at once via asyncio.gather()
    files = {"file": ("sample_text_file.txt", file_bytes, "text/plain")}
    con_data = test_client.post("/upload/", files=files).json()
    print(f"Concurrent: {con_data['chunk_count']} chunks in {con_data['elapsed_seconds']}s")

    speedup = seq_data['elapsed_seconds'] / con_data['elapsed_seconds']
    print(f"\nConcurrent was {speedup:.1f}x faster for {seq_data['chunk_count']} chunks")

    # Query: ask a question answerable from the uploaded document
    question = "What advantage does asyncio.gather provide for document processing?"
    query_data = test_client.post(f"/query?question={question}").json()

print("\nQuestion:", question)
print("\nRetrieved sources:")
print(f"\nAnswer: {query_data['answer']}")

Sequential: 8 chunks in 4.21s
Concurrent: 8 chunks in 0.97s

Concurrent was 4.3x faster for 8 chunks



Question: What advantage does asyncio.gather provide for document processing?

Retrieved sources:

Answer: Based on the provided context, `asyncio.gather` allows multiple requests to launch simultaneously so that the total wall time approaches that of a single request (without blocking the server).


### Optional Exercise: chunk_size=100

Change `chunk_size` from 200 to 100 (this roughly doubles the number of chunks). Embed all chunks both sequentially and concurrently, then compare how each approach scales.